# Classic Notebook
> Classic track note: This notebook demonstrates legacy/classic LangChain-era patterns for evaluation and comparison.
> Prefer the modern equivalents in `lessons/2026-langchain/` for current APIs and recommended techniques.


In [ ]:
%pip install -q langchain langchain-community langchain-openai

In [ ]:
%pip install -q python-dotenv
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import yaml

# https://raw.githubusercontent.com/open-meteo/open-meteo/main/openapi.yml
with open("data/open-meteo.yml") as f:
    raw_meteo_api_spec = yaml.load(f, Loader=yaml.Loader)

# https://github.com/langchain-ai/langchain/issues/2456
# in reduce_openapi_spec
#    servers=spec["servers"],
#            ~~~~^^^^^^^^^^^
raw_meteo_api_spec["servers"] = [{"url": "https://api.open-meteo.com"}]
# raw_meteo_api_spec.servers[0]["url"] = "https://api.foo.com"
# print(raw_meteo_api_spec)

from langchain_community.agent_toolkits.openapi.spec import reduce_openapi_spec
meteo_api_spec = reduce_openapi_spec(raw_meteo_api_spec)


In [ ]:
from _lessonshelper.pretty_print_callback_handler import PrettyPrintCallbackHandler
pretty_callback = PrettyPrintCallbackHandler()

from langchain_openai import OpenAI
#agent_llm = OpenAI(temperature=0, callbacks=[pretty_callback])
agent_llm = OpenAI(temperature=0)

In [ ]:
from langchain_community.utilities.requests import RequestsWrapper
from langchain_community.agent_toolkits.openapi import planner

wrapper = RequestsWrapper()
meteo_agent = planner.create_openapi_agent(
    meteo_api_spec,
    requests_wrapper=wrapper,
    llm=agent_llm,
    allow_dangerous_requests=True,
)

user_query = "What are the current weather conditions in Ghent expressed in Celsius?"
try:
    meteo_result = meteo_agent.invoke({"input": user_query})
    answer = meteo_result["output"] if isinstance(meteo_result, dict) and "output" in meteo_result else meteo_result
except Exception as exc:
    # The planner can occasionally propose endpoints not present in the spec; fall back to direct endpoint call.
    print(f"Planner fallback activated: {exc}")
    fallback_url = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude=51.05&longitude=3.72"
        "&current_weather=true&temperature_unit=celsius"
    )
    answer = wrapper.get(fallback_url)
print(answer)
